In [43]:
import pandas as pd

 -------------------------
 Load data

In [44]:
df = pd.read_csv("sales_dates.csv")

print("===== ORIGINAL DATA =====")
df.head()

===== ORIGINAL DATA =====


,Order_ID,Order_Date,Product,Category,Region,Sales,Quantity
0,1001,2026-01-05,Laptop,Electronics,West,75000,2
1,1002,2026-01-08,Mouse,Accessories,North,1500,5
2,1003,2026-01-12,Keyboard,Accessories,West,3000,3
3,1004,2026-01-18,Monitor,Electronics,South,25000,2
4,1005,2026-02-03,Laptop,Electronics,North,80000,2


 -------------------------
 Convert date

In [45]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"]
)

print("\n===== DATA TYPES =====")
df.dtypes


===== DATA TYPES =====


Order_ID               int64
Order_Date    datetime64[us]
Product                  str
Category                 str
Region                   str
Sales                  int64
Quantity               int64
dtype: object

 -------------------------
 Extract date information

In [51]:
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Day"] = df["Order_Date"].dt.day
df["Day_Name"] = df["Order_Date"].dt.day_name()
df["Weekday"] = df["Order_Date"].dt.weekday
df["Quarter"] = df["Order_Date"].dt.quarter

print("\n===== DATE FEATURES =====")
df[
    [
        "Order_Date",
        "Year",
        "Month",
        "Month_Name",
        "Day_Name",
        "Quarter"
    ]
]



===== DATE FEATURES =====


,Order_Date,Year,Month,Month_Name,Day_Name,Quarter
0,2026-01-05,2026,1,January,Monday,1
1,2026-01-08,2026,1,January,Thursday,1
2,2026-01-12,2026,1,January,Monday,1
3,2026-01-18,2026,1,January,Sunday,1
4,2026-02-03,2026,2,February,Tuesday,1
5,2026-02-09,2026,2,February,Monday,1
6,2026-02-15,2026,2,February,Sunday,1
7,2026-02-22,2026,2,February,Sunday,1
8,2026-03-04,2026,3,March,Wednesday,1
9,2026-03-11,2026,3,March,Wednesday,1


 -------------------------
 Earliest / latest order

In [47]:
print("\n===== DATE RANGE =====")
print("First Order:", df["Order_Date"].min())
print("Last Order:", df["Order_Date"].max())


===== DATE RANGE =====
First Order: 2026-01-05 00:00:00
Last Order: 2026-06-17 00:00:00


 -------------------------
 Days from first order

In [48]:
df["Days_From_First_Order"] = (
    df["Order_Date"]
    - df["Order_Date"].min()
).dt.days

 -------------------------
 Sales by month

In [52]:
monthly_report = (
    df.groupby("Month")
    .agg(
        Total_Sales=("Sales", "sum"),
        Average_Sales=("Sales", "mean"),
        Total_Quantity=("Quantity", "sum"),
        Number_of_Orders=("Order_ID", "count")
    )
    .sort_index()
)

print("\n===== MONTHLY SALES =====")
monthly_report


===== MONTHLY SALES =====


,Total_Sales,Average_Sales,Total_Quantity,Number_of_Orders
Month,,,,
1,104500,26125.000000,12,4
2,118500,29625.000000,11,4
3,105300,26325.000000,15,4
4,76800,25600.000000,9,3
5,110200,36733.333333,8,3
6,72400,36200.000000,5,2


 -------------------------
 Sales by quarter

In [53]:
quarter_sales = (
    df.groupby("Quarter")["Sales"]
    .sum()
)

print("\n===== QUARTERLY SALES =====")
quarter_sales


===== QUARTERLY SALES =====


Quarter
1    328300
2    259400
Name: Sales, dtype: int64

 -------------------------
 Sales by weekday

In [55]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

df["Day_Name"] = pd.Categorical(
    df["Day_Name"],
    categories=weekday_order,
    ordered=True
)

weekday_sales = (
    df.groupby("Day_Name", observed=True)["Sales"]
    .sum()
)

print("\n===== WEEKDAY SALES =====")
weekday_sales


===== WEEKDAY SALES =====


Day_Name
Monday        83000
Tuesday      193400
Wednesday    177700
Thursday      73500
Friday         1600
Sunday        58500
Name: Sales, dtype: int64

 -------------------------
 Date filtering

In [56]:
march_sales = df[
    df["Order_Date"].between(
        "2026-03-01",
        "2026-03-31"
    )
]

print("\n===== MARCH ORDERS =====")
march_sales


===== MARCH ORDERS =====


,Order_ID,Order_Date,Product,Category,Region,Sales,Quantity,Year,Month,Month_Name,Day,Day_Name,Weekday,Quarter,Days_From_First_Order
8,1009,2026-03-04,Mouse,Accessories,North,1800,6,2026,3,March,4,Wednesday,2,1,58
9,1010,2026-03-11,Laptop,Electronics,South,70000,2,2026,3,March,11,Wednesday,2,1,65
10,1011,2026-03-18,Headphones,Accessories,North,5500,5,2026,3,March,18,Wednesday,2,1,72
11,1012,2026-03-25,Monitor,Electronics,South,28000,2,2026,3,March,25,Wednesday,2,1,79


 -------------------------
 Resampling

In [57]:
time_df = df.sort_values(
    "Order_Date"
).set_index("Order_Date")

monthly_resampled = (
    time_df["Sales"]
    .resample("ME")
    .sum()
)

print("\n===== RESAMPLED MONTHLY SALES =====")
monthly_resampled


===== RESAMPLED MONTHLY SALES =====


Order_Date
2026-01-31    104500
2026-02-28    118500
2026-03-31    105300
2026-04-30     76800
2026-05-31    110200
2026-06-30     72400
Freq: ME, Name: Sales, dtype: int64

 -------------------------
 Export

In [58]:
monthly_report.to_csv(
    "monthly_sales_report.csv"
)

print("\nMonthly report exported successfully.")


Monthly report exported successfully.
